# Silver Layer — Data Cleaning and Standardization

Reads from `workspace.bronze` and applies type corrections, explicit null handling, and column standardization. All transformations are documented in DECISIONS.md.

Source: `workspace.bronze.*`
Target: `workspace.silver.*`

In [0]:
CREATE TABLE IF NOT EXISTS workspace.silver.orders
USING DELTA
AS
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date,
    CASE 
        WHEN order_delivered_customer_date IS NOT NULL THEN true
        ELSE false
    END AS is_delivered,
    CASE
        WHEN order_delivered_customer_date IS NOT NULL
        THEN DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date)
        ELSE NULL
    END AS delay_days
FROM workspace.bronze.orders;

In [0]:
SELECT * FROM workspace.silver.orders LIMIT 5;

In [0]:
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN is_delivered = true THEN 1 ELSE 0 END) AS delivered,
    SUM(CASE WHEN is_delivered = false THEN 1 ELSE 0 END) AS not_delivered,
    SUM(CASE WHEN delay_days IS NULL THEN 1 ELSE 0 END) AS null_delay_days
FROM workspace.silver.orders;

In [0]:
CREATE TABLE IF NOT EXISTS workspace.silver.order_items
USING DELTA
AS
SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value
FROM workspace.bronze.order_items;

In [0]:
CREATE TABLE IF NOT EXISTS workspace.silver.order_payments
USING DELTA
AS
SELECT
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
FROM workspace.bronze.order_payments;

In [0]:
CREATE TABLE IF NOT EXISTS workspace.silver.customers
USING DELTA
AS
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM workspace.bronze.customers;

In [0]:
CREATE TABLE IF NOT EXISTS workspace.silver.sellers
USING DELTA
AS
SELECT
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
FROM workspace.bronze.sellers;

In [0]:
CREATE TABLE IF NOT EXISTS workspace.silver.products
USING DELTA
AS
SELECT
    product_id,
    product_category_name,
    product_name_lenght AS product_name_length,
    product_description_lenght AS product_description_length,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
FROM workspace.bronze.products;

In [0]:
CREATE TABLE IF NOT EXISTS workspace.silver.product_category_translation
USING DELTA
AS
SELECT
    product_category_name,
    product_category_name_english
FROM workspace.bronze.product_category_translation;

In [0]:
CREATE TABLE IF NOT EXISTS workspace.silver.geolocation
USING DELTA
AS
SELECT
    geolocation_zip_code_prefix,
    AVG(geolocation_lat) AS geolocation_lat,
    AVG(geolocation_lng) AS geolocation_lng,
    MAX(geolocation_city) AS geolocation_city,
    MAX(geolocation_state) AS geolocation_state
FROM workspace.bronze.geolocation
GROUP BY geolocation_zip_code_prefix;

In [0]:
SELECT COUNT(*) AS total_rows
FROM workspace.silver.geolocation;